## When sharks become intentional bycatch

### This analysis and vizualisation is using the dataset on sharks bycatch from [Worm, Boris et al. (2024)](https://datadryad.org/dataset/doi:10.25349/D9JK6N#readme) to understand what happens to shark caught "unintentionally" by fishers.


### Downloading packages

In [236]:
import pandas as pd                         
import numpy as np    



### Importing datasets

I am looking at the survival rate of each shark species after they were caught as bycatch and released. For this I need bycatch_fate.csv

In [237]:
df = pd.read_csv('bycatch_fate.csv')
df.head()

,species,common_name,family,gear_class,gear_type,fishery_type,target_fishery_species,target_fishery_common,flagstate,ocean,...,fate_proportion,units,sample_size,methods,start_year,end_year,time_scale,reference,notes,reviewer
0,Galeus melastomus,blackmouth catshark,Pentanchidae,longline,NaN,NaN,NaN,NaN,NaN,Atlantic,...,0.533,count,737.0,researcher,1997,1998,annual,Coelho et al 2005,no mention of rfmo,LF
1,Galeus melastomus,blackmouth catshark,Pentanchidae,longline,NaN,NaN,NaN,NaN,NaN,Atlantic,...,0.467,count,646.0,researcher,1997,1998,annual,Coelho et al 2005,no mention of rfmo,LF
2,Etmopterus pusillus,smooth lanternshark,Etmopteridae,longline,NaN,NaN,NaN,NaN,NaN,Atlantic,...,1.000,count,474.0,researcher,1997,1998,annual,Coelho et al 2005,no mention of rfmo,LF
3,Scyliorhinus canicula,small spotted catshark,Scyliorhinidae,longline,NaN,NaN,NaN,NaN,NaN,Atlantic,...,0.897,count,278.0,researcher,1997,1998,annual,Coelho et al 2005,no mention of rfmo,LF
4,Scyliorhinus canicula,small spotted catshark,Scyliorhinidae,longline,NaN,NaN,NaN,NaN,NaN,Atlantic,...,0.103,count,32.0,researcher,1997,1998,annual,Coelho et al 2005,no mention of rfmo,LF


### Exploring the dataset

In [238]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   species                 193 non-null    object 
 1   common_name             193 non-null    object 
 2   family                  193 non-null    object 
 3   gear_class              193 non-null    object 
 4   gear_type               94 non-null     object 
 5   fishery_type            142 non-null    object 
 6   target_fishery_species  42 non-null     object 
 7   target_fishery_common   62 non-null     object 
 8   flagstate               161 non-null    object 
 9   ocean                   164 non-null    object 
 10  ocean_region            155 non-null    object 
 11  location                102 non-null    object 
 12  habitat                 193 non-null    object 
 13  fate_type               193 non-null    object 
 14  fate_proportion         193 non-null    fl

### Checking the different types of sharks

In [239]:
print(df['common_name'].value_counts().head(25))

common_name
blue shark                    16
tiger shark                   11
shortfin mako                 10
silky shark                    9
oceanic whitetip shark         9
sandbar shark                  9
small spotted catshark         7
dusky shark                    7
bigeye thresher shark          7
night shark                    6
scalloped hammerhead           6
blackmouth catshark            5
thornback ray                  4
porbeagle shark                4
scalloped hammerhead shark     3
spiny dogfish                  3
starry smooth-hound            3
school shark                   3
blacktip shark                 3
bull shark                     3
blacknose shark                3
mediterranean starry ray       3
blonde ray                     3
marbled electric ray           3
thresher sharks nei            2
Name: count, dtype: int64


### Check for duplicate species

In [240]:
#looking for unique names

names = (
    df['common_name']
      .dropna()
      .str.lower()
      .str.strip()
      .unique()
)

names = pd.Series(names, name='name')
names

0            blackmouth catshark
1            smooth lanternshark
2         small spotted catshark
3                     blue shark
4                          skate
                 ...            
58        great hammerhead shark
59    scalloped hammerhead shark
60                blacktip shark
61                    bull shark
62               blacknose shark
Name: name, Length: 63, dtype: object

In [241]:
#comparing names that share at least one word

names_df = names.to_frame()
names_df['tokens'] = names_df['name'].str.split()


In [242]:
#looking for the pairs
pairs = []

for i, row in names_df.iterrows():
    a = row['name']
    a_tokens = set(row['tokens'])

    # only compare against names that share at least one token
    candidates = names_df[
        names_df['tokens'].apply(lambda x: bool(a_tokens & set(x)))
    ]['name']

    for b in candidates:
        if a != b and (a in b or b in a):
            pairs.append(tuple(sorted((a, b))))

pairs_df = (
    pd.DataFrame(pairs, columns=['name_1', 'name_2'])
      .drop_duplicates()
      .sort_values(['name_1', 'name_2'])
)

pairs_df


,name_1,name_2
10,bigeye thresher shark,thresher shark
1,cuckoo skate,skate
2,longnose skate,skate
0,longnosed skate,skate
14,scalloped hammerhead,scalloped hammerhead shark
5,shortfin mako,shortfin mako shark
3,skate,white skate
8,thresher shark,thresher sharks nei


### Combining scalloped hammerhead variations and shortfin mako variations into one label (others are truly different species)

In [243]:

mapping = {
    'scalloped hammerhead shark': 'scalloped hammerhead',
    'shortfin mako shark': 'shortfin mako'
}

df['common_name_clean'] = df['common_name'].replace(mapping)


### Dropping rays and renaming unspecified groups (nei)

Rays are a different species from sharks, so dropping them. Names ending in "nei" (not elsewhere included) lump multiple species together, so grouping them into one category and renaming as "other" for clarity.


In [244]:
# Drop rays
df = df[~df['common_name_clean'].str.contains('ray', case=False)]




In [245]:
# Rename nei groups to "other"
df.loc[df['common_name_clean'].str.endswith('nei'), 'common_name_clean'] = 'other'

### Checking the different fates of sharks

In [246]:

print(df['fate_type'].value_counts())

fate_type
discard_unknown     41
discard_dead        36
discard_alive       36
retained_unknown    35
retained_whole      17
Name: count, dtype: int64


### Renaming the fates into three meaningful categories and removing discard unknown because the fate of this category is unknown


In [247]:
def categorize_fate(fate):
    fate = str(fate).lower()

    if 'alive' in fate:
        return 'released'
    if 'dead' in fate and 'discard' in fate:
        return 'dead'
    if 'retained' in fate or 'whole' in fate:
        return 'kept'
    return 'unknown'

df['fate'] = df['fate_type'].apply(categorize_fate)

print(df['fate'].value_counts())


fate
kept        52
unknown     41
dead        36
released    36
Name: count, dtype: int64


In [248]:
#drop the unknown

df = df[df['fate'] != 'unknown']

print(df['fate'].value_counts())

fate
kept        52
dead        36
released    36
Name: count, dtype: int64


### Checking the types of different fishing gear

In [249]:
print(df['gear_class'].value_counts())


gear_class
longline    104
mixed         9
gillnet       7
trawler       4
Name: count, dtype: int64


### Sample size distribution

Each observation comes from a different study. Some studied 6 sharks, some studied 598,000. I am planning to weigh by sample size later so piacking ones with bigger sample sizes.

In [250]:
df['sample_size'].describe()

count       123.000000
mean       1437.170732
std       10589.631235
min           0.000000
25%          11.500000
50%          43.000000
75%         256.000000
max      116172.000000
Name: sample_size, dtype: float64

In [251]:
# Drop rows with missing or zero sample size before analysis
df = df.dropna(subset=['fate_proportion', 'sample_size'])
df = df[df['sample_size'] > 0]



In [252]:
#capitalize the common name
df["common_name_clean"] = df["common_name_clean"].str.title()


### Analysis for chart 1

In [253]:
# Weighted average per species for each fate category separately

def weighted_avg(group):
    return np.average(group['fate_proportion'], weights=group['sample_size'])

released = df[df['fate'] == 'released'].groupby('common_name_clean').apply(weighted_avg, include_groups=False)
dead = df[df['fate'] == 'dead'].groupby('common_name_clean').apply(weighted_avg, include_groups=False)
kept = df[df['fate'] == 'kept'].groupby('common_name_clean').apply(weighted_avg, include_groups=False)








In [254]:
# Combine into one dataframe
chart1 = pd.DataFrame({'released': released, 'dead': dead, 'kept': kept}).fillna(0)
chart1

,released,dead,kept
common_name_clean,,,
Bigeye Thresher Shark,0.414069,0.458250,0.139000
Blackmouth Catshark,0.000000,0.000000,0.421854
Blacknose Shark,0.027000,0.141000,0.622000
Blacktip Shark,0.001000,0.014000,0.968000
Blue Shark,0.607747,0.366286,0.351532
Bluntnose Sixgill Shark,0.000000,0.000000,0.025000
Bull Shark,0.012000,0.009000,0.946000
Common Smooth-Hound,0.000000,0.000000,1.000000
Common Torpedo,0.000000,0.000000,0.970000


In [255]:
# Normalize to 100% 
row_total = chart1.sum(axis=1)
chart1 = chart1.div(row_total, axis=0).multiply(100).round(1)
chart1 = chart1.sort_values('kept', ascending=False)

chart1.to_csv('chart1.csv', index=True)


In [256]:
# Overall % kept 
overall_kept = chart1['kept'].mean()
overall_kept


np.float64(64.80689655172414)

###  Analysis for Chart 2

Checking the percentage 'kept' for each habitat type.

In [257]:
# Group by species in each habitat
species_habitat = df.groupby('common_name_clean')['habitat'].first()
species_habitat

common_name_clean
Bigeye Thresher Shark             pelagic
Blackmouth Catshark              deep-sea
Blacknose Shark             small coastal
Blacktip Shark              large coastal
Blue Shark                       deep-sea
Bluntnose Sixgill Shark          deep-sea
Bull Shark                  large coastal
Common Smooth-Hound              deep-sea
Common Torpedo                   deep-sea
Crocodile Shark                   pelagic
Cuckoo Skate                     deep-sea
Dusky Shark                       pelagic
Great Hammerhead Shark      large coastal
Longnosed Skate                  deep-sea
Night Shark                       pelagic
Nursehound                       deep-sea
Oceanic Whitetip Shark            pelagic
Other                             pelagic
Porbeagle Shark                   pelagic
Sandbar Shark                     pelagic
Scalloped Hammerhead              pelagic
School Shark                     deep-sea
Shortfin Mako                    deep-sea
Silky Shark     

In [258]:
# Adding habitat to chart1 so that I don't have to recalculate species rates
chart2 = chart1.copy()
chart2 = chart2.join(
    pd.Series(species_habitat, name='habitat'),
    on=chart2.index
)

In [259]:
# Average across species within each habitat
chart2 = chart2.groupby('habitat')[['released', 'dead', 'kept']].mean().round(1)


In [260]:
#filtering out 'kept'
chart2 = chart2.sort_values('kept', ascending=True)
chart2

,released,dead,kept
habitat,,,
pelagic,38.3,30.6,31.1
large coastal,0.4,24.2,75.4
small coastal,3.4,17.8,78.7
deep-sea,5.1,2.5,92.4


In [261]:
# Renaming habitats for clarity
chart2.index = chart2.index.astype(str).str.strip().str.lower()

habitat_labels = {
    'pelagic': 'Open Ocean',
    'large coastal': 'Nearshore, Deep Water',
    'small coastal': 'Nearshore, Shallow Water',
    'deep-sea': 'Deep-Sea',
}

chart2 = chart2.rename(index=habitat_labels).sort_values("kept", ascending=False)
chart2

,released,dead,kept
habitat,,,
Deep-Sea,5.1,2.5,92.4
"Nearshore, Shallow Water",3.4,17.8,78.7
"Nearshore, Deep Water",0.4,24.2,75.4
Open Ocean,38.3,30.6,31.1
